# 01: Merge UNSW-NB15 Datasets

This notebook merges the four raw UNSW-NB15 CSV files into a single consolidated dataset and attaches correct column headers derived from the accompanying features metadata file.

The raw data is split across four separate CSV files with no column headers. The features metadata file contains the authoritative ordered list of column names. Without combining these sources, downstream profiling and cleaning steps would operate on an incomplete and unlabelled dataset. The expected output is a single CSV file at `data/processed/UNSW-NB15.csv` containing all records with named columns, ready for data profiling in notebook 02.

## 1.1: Import Libraries

This section imports all libraries required by the notebook and configures global display settings. Only pandas is needed for this notebook as the task is purely loading, combining, and exporting tabular data. The display option prevents pandas from truncating long column values in any intermediate outputs.

In [ ]:
import os
import pandas as pd

# Prevent truncation of long column values in displayed DataFrames.
pd.set_option('display.max_colwidth', None)

## 1.2: Load Features Metadata

This section loads the features metadata file, which contains the ordered list of column names for the raw network data files. The raw CSV files contain no headers, so the correct column names must be sourced from this separate metadata file. The `latin-1` encoding is required because the file contains non-ASCII characters in some feature descriptions. The result is displayed to allow visual verification that all 49 feature names and their attributes have been loaded correctly.

In [ ]:
# Load the features metadata file to extract the ordered list of column names.
# The file uses latin-1 encoding due to non-ASCII characters in feature descriptions.
df_features = pd.read_csv('../data/raw/UNSW-NB15 - Features.csv', na_values=['-'], encoding='latin-1')

In [ ]:
# Display the features metadata to verify all column names loaded correctly.
display(df_features)

## 1.3: Load Network Data

This section loads the four raw network data CSV files. Each file contains a subset of the full dataset with no column headers, so `header=None` is passed to prevent the first data row from being misinterpreted as column names. The `'-'` character is treated as a missing value throughout the pipeline. The `low_memory=False` option is used to ensure pandas infers consistent dtypes across each file rather than processing them in chunks.

In [ ]:
# Load each of the four raw network data CSV files.
# header=None prevents the first data row from being interpreted as column names.
df_nd_1 = pd.read_csv('../data/raw/UNSW-NB15 - 1.csv', na_values=['-'], header=None, low_memory=False)
df_nd_2 = pd.read_csv('../data/raw/UNSW-NB15 - 2.csv', na_values=['-'], header=None, low_memory=False)
df_nd_3 = pd.read_csv('../data/raw/UNSW-NB15 - 3.csv', na_values=['-'], header=None, low_memory=False)
df_nd_4 = pd.read_csv('../data/raw/UNSW-NB15 - 4.csv', na_values=['-'], header=None, low_memory=False)

## 1.4: Combine DataFrames

This section concatenates the four network data DataFrames into a single DataFrame. The `ignore_index=True` argument resets the row index so that each record has a unique sequential index in the merged result. The first five rows are displayed to confirm the raw data structure before column names are applied.

In [ ]:
# Concatenate all four DataFrames into a single merged DataFrame.
df = pd.concat([df_nd_1, df_nd_2, df_nd_3, df_nd_4], ignore_index=True)

In [ ]:
# Display the first five rows to verify the merged structure before column names are applied.
df.head()

## 1.5: Set Column Headers

This section assigns the correct column names to the merged DataFrame using the ordered feature list extracted from the metadata file. The `Name` column of the features metadata contains the 49 feature names in the same order as the columns in the raw data files. After this step, all columns will be accessible by name rather than by numeric index. The first five rows are displayed again to confirm that column names have been applied correctly.

In [ ]:
# Extract the ordered feature names from the metadata and assign them as column headers.
column_headers = df_features["Name"].tolist()
df.columns = column_headers

In [ ]:
# Display the first five rows to confirm column names have been applied correctly.
df.head()

## 1.6: Export Merged Dataset

This section writes the merged and labelled DataFrame to a CSV file at `data/processed/UNSW-NB15.csv`. This file serves as the input for all subsequent notebooks in the pipeline. The index is excluded from the output to keep the file clean and avoid introducing a spurious index column during downstream loads.

In [ ]:
# Create the output directory if it does not already exist.
os.makedirs('../data/processed', exist_ok=True)

# Export the merged DataFrame to CSV. The index is excluded to avoid a spurious index column on reload.
df.to_csv('../data/processed/UNSW-NB15.csv', index=False)